## WP009 — Lineup xG/xA Covariate: Validation

WP008 built and unit-tested the mechanism (leakage-free rolling aggregation, model wiring, `predict.py` mirror). This notebook is the actual question: does it help. Same yardstick as every validation WP since WP003 — pooled RPS and paired-bootstrap gap to Pinnacle closing odds on the 401-match walk-forward comparison, plus the disagreement-decile overfitting check.

Four arms, reusing existing checkpoints where possible — `baseline` (WP001) and `loose_combo` (WP005's finalist) are NOT re-run, only the two new lineup arms actually fit.

In [1]:
import json
import pickle
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from football_model.model.predict import dc_outcome_probs
from football_model.types.model_data import ModelConfig

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'
WP005 = REPO / 'work_products' / 'wp005_prior_loosening'
WP009 = REPO / 'work_products' / 'wp009_lineup_xg_validation'
SCRIPT = REPO / 'scripts' / 'run_cv_window.py'

with open(WP001 / 'cv_shared_data.pkl', 'rb') as f:
    shared_base = pickle.load(f)
df_cv, windows = shared_base['df_cv'], shared_base['windows']

lineup_dev_table = pd.read_pickle(WP009 / 'lineup_dev_table.pkl')
print(len(windows), 'windows;', len(lineup_dev_table), 'lineup_dev_table rows;',
      'teams covered:', sorted(lineup_dev_table['team'].unique()))

# WP009's own shared-data file = WP001's df_cv/windows + the lineup table,
# so run_cv_window.py's shared.get('lineup_dev_table') picks it up.
DATA_PATH = WP009 / 'cv_shared_data.pkl'
if not DATA_PATH.exists():
    pickle.dump({**shared_base, 'lineup_dev_table': lineup_dev_table}, open(DATA_PATH, 'wb'))
    print('wrote', DATA_PATH)

35 windows; 4560 lineup_dev_table rows; teams covered: ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Manchester City', 'Manchester United', 'Newcastle United', 'Norwich', 'Nottingham Forest', 'Sheffield United', 'Southampton', 'Sunderland', 'Tottenham', 'Watford', 'West Bromwich Albion', 'West Ham', 'Wolverhampton Wanderers']


### Candidate configs

In [2]:
LOOSE = dict(init_scale=0.30, home_adv_sd=0.06, sigma_att=0.020, sigma_def=0.020)  # WP005's loose_combo

ARMS = {
    'baseline':             {},
    'loose_combo':          dict(LOOSE),
    'lineup_only':          {'use_lineup_xg': True},
    'lineup_loose_combo':   {'use_lineup_xg': True, **LOOSE},
}
BASE = dict(clip_theta=5.0, center_team_strength=False, use_dixon_coles=True, use_xG=True)
for name, ov in ARMS.items():
    ModelConfig(**BASE, **ov)
print('all', len(ARMS), 'arm configs valid')

all 4 arm configs valid


## Phase 2 — Screening CV (18 windows)

`baseline`/`loose_combo` seeded from WP001/WP005's finished checkpoints filtered to the screening windows; only the 2 new arms actually run (~36 fits).

**Heavy compute — run this yourself.**

In [3]:
SCREEN_WINDOWS = list(range(1, len(windows) + 1, 2))
WINDOW_TIMEOUT = 1200

def load_ckpt(p):
    return pickle.load(open(p, 'rb')) if p.exists() else {'results': [], 'cv_match_predictions': []}

def seed_from(src_checkpoint_path, dest_name):
    dest = WP009 / f'cv_checkpoint_{dest_name}.pkl'
    if dest.exists():
        return
    src = load_ckpt(src_checkpoint_path)
    filt = {'results': [r for r in src['results'] if r['window'] in SCREEN_WINDOWS],
            'cv_match_predictions': [m for m in src['cv_match_predictions'] if m['window'] in SCREEN_WINDOWS]}
    pickle.dump(filt, open(dest, 'wb'))
    print(f'seeded {dest_name} from {src_checkpoint_path.name}:', len(filt['results']), 'windows')

seed_from(WP001 / 'cv_checkpoint.pkl', 'baseline')
seed_from(WP005 / 'cv_checkpoint_full_loose_combo.pkl', 'loose_combo')

for arm, ov in ARMS.items():
    if arm in ('baseline', 'loose_combo'):
        continue
    ckpt = WP009 / f'cv_checkpoint_{arm}.pkl'
    done = {r['window'] for r in load_ckpt(ckpt)['results']}
    todo = [w for w in SCREEN_WINDOWS if w not in done]
    print(f'\n### arm {arm}  overrides={ov}  ({len(done)}/{len(SCREEN_WINDOWS)} done, {len(todo)} to run)')
    for w in todo:
        print(f'  [{arm}] window {w}')
        try:
            subprocess.run(
                [sys.executable, str(SCRIPT), '--data-path', str(DATA_PATH),
                 '--checkpoint-path', str(ckpt), '--window-index', str(w),
                 '--config-json', json.dumps(ov)],
                timeout=WINDOW_TIMEOUT, check=True,
            )
        except subprocess.TimeoutExpired:
            print(f'  [{arm}] window {w} TIMEOUT — skipped, re-run cell to retry')
        except subprocess.CalledProcessError:
            print(f'  [{arm}] window {w} FAILED — skipped, re-run cell to retry')

print('\nscreening run complete')
for arm in ARMS:
    n = len(load_ckpt(WP009 / f'cv_checkpoint_{arm}.pkl')['results'])
    print(f'  {arm}: {n}/{len(SCREEN_WINDOWS)}')


### arm lineup_only  overrides={'use_lineup_xg': True}  (18/18 done, 0 to run)

### arm lineup_loose_combo  overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02}  (13/18 done, 5 to run)
  [lineup_loose_combo] window 27
[window 27/35] training rounds 1-166 (use_xg=True, use_dc=True, overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:25<07:27,  8.49it/s]

Running chain 1:   5%|▌         | 200/4000 [00:26<07:48,  8.11it/s]

Running chain 0:  10%|█         | 400/4000 [00:33<04:14, 14.15it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:39<02:59, 18.98it/s]

Running chain 0:  20%|██        | 800/4000 [00:45<02:19, 22.86it/s]

Running chain 0:  25%|██▌       | 1000/4000 [00:51<01:56, 25.76it/s]

Running chain 0:  30%|███       | 1200/4000 [00:57<01:39, 28.19it/s]

Running chain 0:  35%|███▌      | 1400/4000 [01:02<01:26, 29.94it/s]

Running chain 0:  40%|████      | 1600/4000 [01:08<01:16, 31.18it/s]

Running chain 0:  45%|████▌     | 1800/4000 [01:14<01:08, 32.10it/s]

Running chain 0:  50%|█████     | 2000/4000 [01:20<01:02, 32.05it/s]

Running chain 0:  55%|█████▌    | 2200/4000 [01:26<00:55, 32.71it/s]

Running 

[window 27] MAE=1.100 LL_improvement=3.66
[window 27] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_lineup_loose_combo.pkl
  [lineup_loose_combo] window 29
[window 29/35] training rounds 1-176 (use_xg=True, use_dc=True, overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:36<10:52,  5.82it/s]

Running chain 0:  10%|█         | 400/4000 [00:48<06:14,  9.62it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:56<04:12, 13.49it/s]

Running chain 0:  20%|██        | 800/4000 [01:04<03:15, 16.35it/s]

Running chain 0:  25%|██▌       | 1000/4000 [01:13<02:41, 18.55it/s]

Running chain 0:  30%|███       | 1200/4000 [01:20<02:16, 20.49it/s]

Running chain 0:  35%|███▌      | 1400/4000 [01:28<01:59, 21.77it/s]

Running chain 0:  40%|████      | 1600/4000 [01:36<01:46, 22.64it/s]

Running chain 0:  45%|████▌     | 1800/4000 [01:45<01:35, 23.10it/s]

Running chain 0:  50%|█████     | 2000/4000 [01:53<01:24, 23.61it/s]

Running chain 0:  55%|█████▌    | 2200/4000 [02:01<01:14, 24.16it/s]

Running chain 0:  60%|██████    | 2400/4000 [02:09<01:05, 24.30it/s]

Running

[window 29] MAE=0.575 LL_improvement=-0.09
[window 29] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_lineup_loose_combo.pkl
  [lineup_loose_combo] window 31
[window 31/35] training rounds 1-186 (use_xg=True, use_dc=True, overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


...

[window 31] MAE=1.071 LL_improvement=1.33
[window 31] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_lineup_loose_combo.pkl
  [lineup_loose_combo] window 33
[window 33/35] training rounds 1-196 (use_xg=True, use_dc=True, overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:37<11:29,  5.51it/s]

Running chain 1:  10%|█         | 400/4000 [00:48<06:16,  9.55it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:51<03:58, 14.27it/s]

Running chain 0:  20%|██        | 800/4000 [01:00<03:10, 16.80it/s]

Running chain 0:  25%|██▌       | 1000/4000 [01:08<02:40, 18.70it/s][A

Running chain 0:  30%|███       | 1200/4000 [01:17<02:19, 20.02it/s]

Running chain 0:  35%|███▌      | 1400/4000 [01:25<02:02, 21.17it/s]

Running chain 0:  40%|████      | 1600/4000 [01:34<01:50, 21.66it/s]

Running chain 0:  45%|████▌     | 1800/4000 [01:43<01:40, 21.92it/s]

Running chain 0:  50%|█████     | 2000/4000 [01:52<01:30, 22.12it/s]

Running chain 0:  55%|█████▌    | 2200/4000 [02:00<01:19, 22.52it/s]

Running chain 0:  60%|██████    | 2400/4000 [02:09<01:09, 22.86it/s]

Runni

[window 33] MAE=0.836 LL_improvement=-0.00
[window 33] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_lineup_loose_combo.pkl
  [lineup_loose_combo] window 35
[window 35/35] training rounds 1-206 (use_xg=True, use_dc=True, overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:43<13:12,  4.79it/s]

Running chain 0:  10%|█         | 400/4000 [00:54<07:09,  8.38it/s]

Running chain 0:  15%|█▌        | 600/4000 [01:04<04:55, 11.51it/s]

Running chain 0:  20%|██        | 800/4000 [01:15<03:56, 13.54it/s]

Running chain 0:  20%|██        | 800/4000 [01:30<03:56, 13.54it/s]]

Running chain 0:  25%|██▌       | 1000/4000 [01:41<04:46, 10.47it/s]

Running chain 0:  30%|███       | 1200/4000 [01:53<03:49, 12.18it/s]

Running chain 0:  35%|███▌      | 1400/4000 [02:02<03:04, 14.08it/s]

Running chain 0:  40%|████      | 1600/4000 [02:12<02:33, 15.63it/s]

Running chain 0:  45%|████▌     | 1800/4000 [02:23<02:15, 16.23it/s]

Running chain 0:  50%|█████     | 2000/4000 [02:32<01:53, 17.68it/s]

Running chain 0:  

[window 35] MAE=0.922 LL_improvement=0.67
[window 35] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_lineup_loose_combo.pkl

screening run complete
  baseline: 18/18
  loose_combo: 18/18
  lineup_only: 18/18
  lineup_loose_combo: 18/18


### Phase 2 analysis — resolution + gap to Pinnacle per arm

In [4]:
odds_raw = pd.read_pickle(WP003 / 'odds_raw.pkl')
CODE_TO_FD = {'ARS': 'Arsenal', 'AVL': 'Aston Villa', 'BOU': 'Bournemouth', 'BRE': 'Brentford',
    'BRI': 'Brighton', 'BUR': 'Burnley', 'CHE': 'Chelsea', 'CRY': 'Crystal Palace', 'EVE': 'Everton',
    'FLH': 'Fulham', 'IPS': 'Ipswich', 'LED': 'Leeds', 'LEI': 'Leicester', 'LIV': 'Liverpool',
    'LUT': 'Luton', 'MCI': 'Man City', 'MUN': 'Man United', 'NEW': 'Newcastle', 'NOR': 'Norwich',
    'NOT': "Nott'm Forest", 'SHE': 'Sheffield United', 'SOU': 'Southampton', 'SUN': 'Sunderland',
    'TOT': 'Tottenham', 'WAT': 'Watford', 'WBA': 'West Brom', 'WHU': 'West Ham', 'WOL': 'Wolves'}
df_sorted = df_cv.sort_values('datetime').reset_index(drop=True)

def fixtures_for(ckpt, windows_list=None):
    windows_list = windows if windows_list is None else windows_list
    mp = ckpt['cv_match_predictions']
    rows = []
    for w in sorted({m['window'] for m in mp}):
        win = windows_list[w - 1]
        sel = df_sorted[(df_sorted['is_home'] == 1) & (df_sorted['round'] >= win['test_start'])
                        & (df_sorted['round'] <= win['test_end'])]
        for (_, r), m in zip(sel.iterrows(), [x for x in mp if x['window'] == w]):
            assert int(r['goals_home']) == m['goals_home'] and int(r['goals_away']) == m['goals_away']
            rows.append({'date': pd.Timestamp(r['datetime']).normalize(),
                         'home_fd': CODE_TO_FD[r['team']], 'away_fd': CODE_TO_FD[r['opp_team']],
                         'lambda_home': m['lambda_home'], 'lambda_away': m['lambda_away'],
                         'rho_dc': m.get('rho_dc'), 'goals_home': m['goals_home'], 'goals_away': m['goals_away']})
    d = pd.DataFrame(rows)
    P = np.array([dc_outcome_probs(x.lambda_home, x.lambda_away, rho=x.rho_dc) for x in d.itertuples()])
    d[['p_home', 'p_draw', 'p_away']] = P
    d['result'] = np.where(d['goals_home'] > d['goals_away'], 'H',
                    np.where(d['goals_home'] == d['goals_away'], 'D', 'A'))
    return d.merge(odds_raw, left_on=['date', 'home_fd', 'away_fd'],
                   right_on=['Date', 'HomeTeam', 'AwayTeam'], how='inner')

def rps_row(ph, pdw, pa, a):
    c1, c2 = ph, ph + pdw
    e1 = 1.0 if a == 'H' else 0.0
    e2 = 1.0 if a in ('H', 'D') else 0.0
    return 0.5 * ((c1 - e1) ** 2 + (c2 - e2) ** 2)

def boot(v, n=4000, seed=0):
    v = np.asarray(v, float); rng = np.random.default_rng(seed)
    b = np.array([rng.choice(v, len(v), replace=True).mean() for _ in range(n)])
    return v.mean(), *np.percentile(b, [2.5, 97.5])

def devig(o):
    inv = 1 / np.asarray(o, float); return inv / inv.sum()

rows = []
for arm in ARMS:
    ck = WP009 / f'cv_checkpoint_{arm}.pkl'
    if not ck.exists() or len(load_ckpt(ck)['results']) == 0:
        print(f'{arm}: not run yet'); continue
    d = fixtures_for(load_ckpt(ck))
    pin_ok = d[['PSCH', 'PSCD', 'PSCA']].notna().all(axis=1) if 'PSCH' in d else pd.Series(False, index=d.index)
    d = d[pin_ok].copy()
    Pin = np.array([devig(r) for r in d[['PSCH', 'PSCD', 'PSCA']].to_numpy()])
    d['rps_m'] = [rps_row(x.p_home, x.p_draw, x.p_away, x.result) for x in d.itertuples()]
    d['rps_p'] = [rps_row(Pin[i, 0], Pin[i, 1], Pin[i, 2], d['result'].iloc[i]) for i in range(len(d))]
    d['disag'] = np.abs(d['p_home'].values - Pin[:, 0])
    g_all, lo, hi = boot((d['rps_m'] - d['rps_p']).values)
    top = d[d['disag'] >= d['disag'].quantile(0.75)]
    g_top, _, _ = boot((top['rps_m'] - top['rps_p']).values)
    rows.append({'arm': arm, 'n': len(d), 'model RPS': round(d['rps_m'].mean(), 4),
                 'gap vs Pinnacle': f'{g_all:+.4f}', 'CI': f'[{lo:+.4f},{hi:+.4f}]',
                 'gap top-25% disagree': f'{g_top:+.4f}'})
print(pd.DataFrame(rows).to_string(index=False))

               arm   n  model RPS gap vs Pinnacle                CI gap top-25% disagree
          baseline 195     0.1916         +0.0129 [+0.0058,+0.0200]              +0.0455
       loose_combo 195     0.1905         +0.0118 [+0.0049,+0.0188]              +0.0400
       lineup_only 195     0.1915         +0.0128 [+0.0058,+0.0197]              +0.0434
lineup_loose_combo 195     0.1903         +0.0117 [+0.0049,+0.0186]              +0.0453


### Phase 2b — does fitted `beta_lineup` behave sensibly?

Not a "does it help" check (that's the table above) — a sanity check that the coefficient itself is well-identified (not pinned at the prior's edge, not blown up) across the screening windows.

In [5]:
for arm in ['lineup_only', 'lineup_loose_combo']:
    cp = load_ckpt(WP009 / f'cv_checkpoint_{arm}.pkl')
    vals = [r['beta_lineup'] for r in cp['results'] if r.get('beta_lineup') is not None]
    if not vals:
        print(f'{arm}: no beta_lineup recorded (not run yet)'); continue
    vals = np.array(vals)
    print(f'{arm}: beta_lineup across {len(vals)} windows -- mean {vals.mean():.4f}  '
          f'range [{vals.min():.4f}, {vals.max():.4f}]  sd {vals.std():.4f}')

lineup_only: beta_lineup across 18 windows -- mean 0.2148  range [0.1422, 0.3054]  sd 0.0548
lineup_loose_combo: beta_lineup across 18 windows -- mean 0.1968  range [0.1306, 0.2859]  sd 0.0529


## Phase 3 — Confirmation (finalist only)

Set `FINALIST` after reading Phase 2, run the full 35-window CV, then the cold held-out-2025-26-season check.

**Heavy compute — run yourself.**

In [7]:
FINALIST = 'lineup_loose_combo'   # <- set to 'lineup_only' or 'lineup_loose_combo' after Phase 2

FULL_WINDOWS = list(range(1, len(windows) + 1))
if FINALIST:
    ov = ARMS[FINALIST]
    ckpt = WP009 / f'cv_checkpoint_full_{FINALIST}.pkl'
    done = {r['window'] for r in load_ckpt(ckpt)['results']}
    for w in [x for x in FULL_WINDOWS if x not in done]:
        print(f'[full {FINALIST}] window {w}')
        try:
            subprocess.run([sys.executable, str(SCRIPT), '--data-path', str(DATA_PATH),
                            '--checkpoint-path', str(ckpt), '--window-index', str(w),
                            '--config-json', json.dumps(ov)], timeout=WINDOW_TIMEOUT, check=True)
        except subprocess.SubprocessError as e:
            print(f'  window {w} problem ({type(e).__name__}) — re-run to retry')
    print('finalist full-CV run complete')
else:
    print('set FINALIST first')

[full lineup_loose_combo] window 1


/Users/hadiahmed/Documents/projects/football-predictor/venv/lib/python3.13/site-packages/arviz/__init__.py:39: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


[window 1/35] training rounds 1-36 (use_xg=True, use_dc=True, overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:06<01:40, 37.90it/s]

Running chain 0:  10%|█         | 400/4000 [00:08<00:51, 70.02it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:09<00:34, 98.26it/s]

Running chain 0:  20%|██        | 800/4000 [00:10<00:26, 118.84it/s][A

Running chain 0:  25%|██▌       | 1000/4000 [00:11<00:21, 138.41it/s][A

Running chain 0:  30%|███       | 1200/4000 [00:12<00:18, 150.51it/s]

Running chain 0:  35%|███▌      | 1400/4000 [00:13<00:15, 164.69it/s]

Running chain 0:  40%|████      | 1600/4000 [00:14<00:13, 177.82it/s]

Running chain 1:  40%|████      | 1600/4000 [00:15<00:14, 161.23it/s]

Running chain 1:  45%|████▌     | 1800/4000 [00:16<00:13, 166.89it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [00:18<00:09, 189.74it/s]

Running chain 1:  65%|██████▌   | 2600/4000 [00:20<00:06, 215.23i

[window 1] MAE=1.157 LL_improvement=1.05
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_full_lineup_loose_combo.pkl
[full lineup_loose_combo] window 2
[window 2/35] training rounds 1-41 (use_xg=True, use_dc=True, overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


...

[window 33] MAE=0.836 LL_improvement=-0.00
[window 33] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_full_lineup_loose_combo.pkl
[full lineup_loose_combo] window 34
[window 34/35] training rounds 1-201 (use_xg=True, use_dc=True, overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:40<12:19,  5.14it/s]

Running chain 1:  10%|█         | 400/4000 [00:49<06:21,  9.43it/s]

Running chain 1:  15%|█▌        | 600/4000 [00:58<04:28, 12.66it/s]

Running chain 2:  10%|█         | 400/4000 [01:00<06:24,  9.35it/s]

Running chain 0:  25%|██▌       | 1000/4000 [01:14<02:54, 17.21it/s][A

Running chain 0:  30%|███       | 1200/4000 [01:23<02:30, 18.57it/s]

Running chain 0:  35%|███▌      | 1400/4000 [01:32<02:14, 19.37it/s]

Running chain 0:  40%|████      | 1600/4000 [01:41<01:59, 20.13it/s]

Running chain 0:  45%|████▌     | 1800/4000 [01:51<01:46, 20.58it/s]

Running chain 0:  50%|█████     | 2000/4000 [02:00<01:36, 20.83it/s]

Running chain 0:  55%|█████▌    | 2200/4000 [02:09<01:25, 21.14it/s]

Running chain 0:  60%|██████    | 2400/4000 [02:19<01:15, 21.08it/s]

Runni

[window 34] MAE=0.861 LL_improvement=0.87
[window 34] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_full_lineup_loose_combo.pkl
[full lineup_loose_combo] window 35
[window 35/35] training rounds 1-206 (use_xg=True, use_dc=True, overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:45<13:49,  4.58it/s]

Running chain 1:  10%|█         | 400/4000 [00:55<07:15,  8.26it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:57<04:23, 12.91it/s]

Running chain 1:  20%|██        | 800/4000 [01:14<03:47, 14.09it/s]

Running chain 0:  25%|██▌       | 1000/4000 [01:15<02:55, 17.12it/s]

Running chain 1:  30%|███       | 1200/4000 [01:33<02:38, 17.61it/s]

Running chain 1:  35%|███▌      | 1400/4000 [01:42<02:17, 18.85it/s]

Running chain 1:  40%|████      | 1600/4000 [01:51<02:02, 19.52it/s]

Running chain 1:  45%|████▌     | 1800/4000 [02:00<01:47, 20.39it/s]

Running chain 1:  50%|█████     | 2000/4000 [02:10<01:36, 20.78it/s]

Running chain 0:  60%|██████    | 2400/4000 [02:19<01:14, 21.56it/s]

Running chain 1:  

[window 35] MAE=0.922 LL_improvement=0.67
[window 35] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_full_lineup_loose_combo.pkl
finalist full-CV run complete


### Phase 3 — held-out-season check

In [8]:
last_season = sorted(df_cv['season'].unique())[-1]
train_end_round = int(df_cv[df_cv['season'] != last_season]['round'].max())
test_rounds = sorted(df_cv[df_cv['season'] == last_season]['round'].unique())
holdout_window = {'train_start': 1, 'train_end': train_end_round,
                  'test_start': int(test_rounds[0]), 'test_end': int(test_rounds[-1])}
print('held-out window:', holdout_window, f'({len(test_rounds)} rounds of {last_season})')

ho_data = WP009 / 'holdout_shared_data.pkl'
if not ho_data.exists():
    pickle.dump({'df_cv': df_cv, 'windows': [holdout_window], 'lineup_dev_table': lineup_dev_table}, open(ho_data, 'wb'))

arms_to_run = ['baseline'] + ([FINALIST] if FINALIST else [])
for arm in arms_to_run:
    ov = ARMS[arm]
    ckpt = WP009 / f'cv_checkpoint_holdout_{arm}.pkl'
    if load_ckpt(ckpt)['results']:
        print(f'{arm}: holdout already done'); continue
    print(f'[holdout {arm}]')
    subprocess.run([sys.executable, str(SCRIPT), '--data-path', str(ho_data),
                    '--checkpoint-path', str(ckpt), '--window-index', '1',
                    '--config-json', json.dumps(ov)], timeout=WINDOW_TIMEOUT, check=True)

for arm in arms_to_run:
    ck = WP009 / f'cv_checkpoint_holdout_{arm}.pkl'
    if not load_ckpt(ck)['results']:
        continue
    d = fixtures_for(load_ckpt(ck), windows_list=[holdout_window])
    pin_ok = d[['PSCH', 'PSCD', 'PSCA']].notna().all(axis=1)
    d = d[pin_ok].copy()
    Pin = np.array([devig(r) for r in d[['PSCH', 'PSCD', 'PSCA']].to_numpy()])
    rm = np.array([rps_row(x.p_home, x.p_draw, x.p_away, x.result) for x in d.itertuples()])
    rp = np.array([rps_row(Pin[i, 0], Pin[i, 1], Pin[i, 2], d['result'].iloc[i]) for i in range(len(d))])
    g, lo, hi = boot(rm - rp)
    print(f'{arm:>22}  n={len(d)}  model RPS {rm.mean():.4f}  gap vs Pinnacle {g:+.4f}  CI [{lo:+.4f},{hi:+.4f}]')

held-out window: {'train_start': 1, 'train_end': 172, 'test_start': 173, 'test_end': 208} (36 rounds of 2025)
[holdout baseline]
[window 1/1] training rounds 1-172 (use_xg=True, use_dc=True, overrides={})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:34<10:16,  6.17it/s]

Running chain 1:  10%|█         | 400/4000 [00:42<05:33, 10.81it/s]

Running chain 1:  15%|█▌        | 600/4000 [00:51<03:59, 14.19it/s]

Running chain 1:  20%|██        | 800/4000 [01:00<03:10, 16.80it/s]

Running chain 1:  25%|██▌       | 1000/4000 [01:09<02:44, 18.26it/s]

Running chain 1:  30%|███       | 1200/4000 [01:18<02:22, 19.68it/s]

Running chain 1:  35%|███▌      | 1400/4000 [01:27<02:06, 20.50it/s]

Running chain 1:  40%|████      | 1600/4000 [01:35<01:52, 21.40it/s]

Running chain 1:  45%|████▌     | 1800/4000 [01:43<01:39, 22.13it/s]

Running chain 1:  50%|█████     | 2000/4000 [01:52<01:29, 22.35it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [02:01<01:19, 22.62it/s]

Running chain 1:  60%|██████    | 2400/4000 [02:09<01:09, 22.97it/s]

Running

[window 1] MAE=0.897 LL_improvement=25.92
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_holdout_baseline.pkl
[holdout lineup_loose_combo]
[window 1/1] training rounds 1-172 (use_xg=True, use_dc=True, overrides={'use_lineup_xg': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:35<10:40,  5.93it/s]

Running chain 0:  10%|█         | 400/4000 [00:49<06:13,  9.63it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:56<04:12, 13.46it/s]

Running chain 0:  20%|██        | 800/4000 [01:04<03:14, 16.42it/s]

Running chain 1:  25%|██▌       | 1000/4000 [01:09<02:38, 18.87it/s]

Running chain 1:  30%|███       | 1200/4000 [01:17<02:15, 20.60it/s]

Running chain 1:  35%|███▌      | 1400/4000 [01:24<01:57, 22.18it/s]

Running chain 1:  40%|████      | 1600/4000 [01:32<01:43, 23.20it/s]

Running chain 1:  45%|████▌     | 1800/4000 [01:40<01:31, 24.04it/s]

Running chain 1:  50%|█████     | 2000/4000 [01:47<01:20, 24.78it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [01:55<01:11, 25.23it/s]

Running chain 1:  

[window 1] MAE=0.888 LL_improvement=29.50
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp009_lineup_xg_validation/cv_checkpoint_holdout_lineup_loose_combo.pkl
              baseline  n=210  model RPS 0.2017  gap vs Pinnacle +0.0023  CI [-0.0048,+0.0098]
    lineup_loose_combo  n=210  model RPS 0.2001  gap vs Pinnacle +0.0008  CI [-0.0062,+0.0077]


## Results

See `README.md` for the writeup once Phases 2–3 are run.